# Translation

- run diffusion model with classifier guidance

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("anomaly-detection/diffusion-anomaly")
sys.path.append("anomaly-detection/diffusion-anomaly/scripts")

import argparse
import os
from tqdm import tqdm

from torch.autograd import Variable
from guided_diffusion.bratsloader import BRATSDataset
import blobfile as bf
import torch as th
os.environ['OMP_NUM_THREADS'] = '8'

import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel.distributed import DistributedDataParallel as DDP
from torch.optim import AdamW
import numpy as np

from guided_diffusion import dist_util, logger
from guided_diffusion.fp16_util import MixedPrecisionTrainer
from guided_diffusion.image_datasets import load_data, ShuffleDataset
from guided_diffusion.train_util import visualize
from guided_diffusion.validation_plots import validation_plots
from guided_diffusion.resample import create_named_schedule_sampler
from guided_diffusion.script_util import (
    add_dict_to_argparser,
    args_to_dict,
    classifier_and_diffusion_defaults,
    create_classifier_and_diffusion,
    model_and_diffusion_defaults,
    create_model_and_diffusion,
)
from guided_diffusion.train_util import parse_resume_step_from_filename, log_loss_dict
from scripts.classifier_train import compute_top_k

from matplotlib import pyplot as plt

In [ ]:
!ls /scratch/7DayLifetime/munjung/anomaly-detection/npz/erase/

## inspect images

In [ ]:
# load samples to translate

def load_images(path, charge_scale=1.0, show_plot=False):
    numpy_img = np.load(path)
    arr_reco = visualize(numpy_img["reco"]).astype(np.float32)
    arr_truth = numpy_img["truth"].astype(np.float32)

    # weights from truth 
    weights = np.sum(arr_truth, axis=(1, 2, 3)).astype(np.float32)
    weights = weights / np.mean(weights)

    charge_norm = np.mean(arr_truth)
    pixel_weights = 1 + arr_truth / charge_norm / charge_scale
    pixel_weights = pixel_weights / np.mean(pixel_weights)
    charge = np.sum(arr_truth, axis=(1, 2, 3)).astype(np.float32)

    if show_plot:
        for img_idx in range(len(arr_reco)):
            this_img = arr_reco[img_idx][0]
            this_w = weights[img_idx]
            this_pw = pixel_weights[img_idx]
            this_c = charge[img_idx]

            # only plot images with particle tracks
            if np.max(this_img) < 1:
                continue
            plt.imshow(this_img) #, vmin=-100, vmax=100)
            plt.title(f"Image {img_idx}")
            plt.colorbar()
            plt.show()

    return arr_reco, arr_truth, weights, pixel_weights, charge

In [ ]:
path = "/scratch/7DayLifetime/munjung/anomaly-detection/npz/erase/" + "tpc0_plane0_rec_67583145_18_diseased_erase.npz"
filename=str(path).split("/")[-1].split(".")[0]
load_images(path, show_plot=True)

## load networks

In [ ]:
# diffusion model

# model configs -- must match training configs
args = {
    "batch_size": 1,
    "image_size": 512,
    "num_channels": 32,
    "class_cond": False,
    "num_res_blocks": 2,
    "num_heads": 8,
    "learn_sigma": True,
    "use_scale_shift_norm": False,
    "attention_resolutions": "16,32",
    "dataset": "",
    "data_dir": "/scratch/7DayLifetime/munjung/anomaly-detection/npz",
    "learn_sigma":False,
    "diffusion_steps":1000,
    "noise_schedule":"linear",
    "timestep_respacing":"",
    "use_kl":False,
    "predict_xstart":False,
    "predict_xprev":False,
    "rescale_timesteps":False,
    "rescale_learned_sigmas":False,
    "num_heads_upsample":-1,
    "num_head_channels":-1,
    "channel_mult":"",
    "dropout":0.0,
    "use_checkpoint":False,
    "resblock_updown":True,
    "use_fp16":False,
    "use_new_attention_order":False,
}

model, diffusion = create_model_and_diffusion(
        **{k: v for k, v in args.items() if k in model_and_diffusion_defaults()},
)

model_path = "/home/munjung/anomaly-detection/diffusion-anomaly/results/noise_300/emabrats2update_0.9999_030000.pt"
model.load_state_dict(
    dist_util.load_state_dict(model_path, map_location="cpu")
)
model.to(dist_util.dev())
if args["use_fp16"]:
    model.convert_to_fp16()
model.eval()

p1 = np.array([np.array(p.shape).prod() for p in model.parameters()]).sum()

In [ ]:
# classifier model

# model configs -- must match training configs
class_args = {
    "lr": 1e-4,
    "batch_size": 10,
    "image_size": 512,
    "classifier_attention_resolutions": "16,32",
    "classifier_depth": 4,
    "classifier_width": 32,
    "classifier_pool": "attention",
    "classifier_resblock_updown": True,
    "classifier_use_scale_shift_norm": False,
    "classifier_use_fp16": False,
    "learn_sigma": False,
    "noised": False,
    "diffusion_steps": 1000,
    "noise_schedule": "linear",
    "timestep_respacing": "",
    "use_kl": False,
    "predict_xstart": False,
    "predict_xprev": False,
    "rescale_timesteps": False,
    "rescale_learned_sigmas": False,
    "dataset": "",
    "data_dir": "/scratch/7DayLifetime/munjung/anomaly-detection/npz",

}

dist_util.setup_dist()
logger.configure()

logger.log("creating classifier model...")
model_classifier, diffusion_classifier = create_classifier_and_diffusion(
    **{k: v for k, v in class_args.items() if k in classifier_and_diffusion_defaults()},
)
model_classifier.to(dist_util.dev())
if class_args["noised"]:
    schedule_sampler = create_named_schedule_sampler(
        class_args.schedule_sampler, diffusion, maxt=1000
    )

model_classifier_name = "/home/munjung/anomaly-detection/diffusion-anomaly/results/classifier_model_015900.pt"
logger.log(
    f"loading model from checkpoint: {model_classifier_name}..."
)
model_classifier.load_state_dict(
    dist_util.load_state_dict(
        model_classifier_name, map_location=dist_util.dev()
    )
)

if class_args["classifier_use_fp16"]:
    model_classifier.convert_to_fp16()

model_classifier.eval()
dist_util.sync_params(model_classifier.parameters())

## Run translation and save outputs

In [ ]:
def validation_plots_wclass(
    diffusion,
    model,
    basedir,
    img0,
    ddpm=True,
    classifier=None,
    classifier_scale=100.0,
    target_class=0,
):
    save_data = {}

    genf = diffusion.p_sample_loop_progressive if ddpm else diffusion.ddim_sample_loop_progressive
    i0 = img0.cpu().numpy()
    img0 = img0.to(dist_util.dev())

    model_kwargs = {}
    cond_fn = None
    if classifier is not None:
        model_kwargs["y"] = th.tensor([target_class], device=dist_util.dev(), dtype=th.long)

        def cond_fn(x, t, y=None):
            assert y is not None
            with th.enable_grad():
                x_in = x.detach().requires_grad_(True)
                logits = classifier(x_in, t)
                log_probs = F.log_softmax(logits, dim=-1)
                selected = log_probs[th.arange(len(logits), device=logits.device), y.view(-1)]
                grad = th.autograd.grad(selected.sum(), x_in)[0]
                return grad, grad * classifier_scale

    idiffs = []
    ts = []

    plt.imshow(np.squeeze(i0), vmin=-1, vmax=1)
    plt.colorbar()
    plt.savefig(basedir + "img.png", bbox_inches="tight")
    plt.title("Original Image")
    plt.close()

    save_data["original"] = np.squeeze(i0)

    for i, t in enumerate(range(0, 550, 50)):
        if i == 0:
            continue

        t = th.tensor(t - 1, device=dist_util.dev())
        idiff = diffusion.q_sample(img0, t)
        idiffs.append(idiff)
        ts.append(t)

    for i, (idiff, t) in enumerate(zip(idiffs, ts)):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        c1 = ax1.imshow(np.squeeze(idiff.cpu().numpy()))
        fig.colorbar(c1, ax=ax1)

        c2 = ax2.imshow(np.squeeze(idiff.cpu().numpy()) - np.squeeze(i0))
        fig.colorbar(c2, ax=ax2)

        plt.title("Diffused T: %i" % int(t))
        plt.savefig(basedir + "diffusion_t%i.png" % int(t), bbox_inches="tight")
        plt.close()

    gens = []
    for (t, idiff) in zip(ts, idiffs):
        gen = genf(
            model,
            i0.shape,
            time=t.item(),
            noise=idiff.unsqueeze(0),
            cond_fn=cond_fn,
            model_kwargs=model_kwargs,
        )
        final = None
        for g in gen:
            final = g
        gens.append(final)

    for i, (idiff, g, t) in enumerate(zip(idiffs, gens, ts)):
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(16, 4))
        c1 = ax1.imshow(np.squeeze(idiff.cpu().numpy()), vmin=-1, vmax=1)
        fig.colorbar(c1, ax=ax1)
        ax1.title.set_text("Diffused")

        c2 = ax2.imshow(np.squeeze(g["sample"].cpu().numpy()), vmin=-1, vmax=1)
        fig.colorbar(c2, ax=ax2)
        ax2.title.set_text("Reconstructed")

        c3 = ax3.imshow(np.squeeze(g["sample"].cpu().numpy()) - np.squeeze(i0))
        fig.colorbar(c3, ax=ax3)
        ax3.title.set_text("Reconstructed - Original")

        c4 = ax4.imshow(np.squeeze(g["sample"].cpu().numpy()) - np.squeeze(idiff.cpu().numpy()))
        fig.colorbar(c4, ax=ax4)
        ax4.title.set_text("Reconstructed - Diffused")
        plt.savefig(basedir + "reconstructed_t%i.png" % int(t), bbox_inches="tight")
        plt.close()

        step_result = {
            "diffused": idiff.cpu().numpy(),
            "reconstructed": g["sample"].cpu().numpy(),
            "t": t,
        }
        if "saliency" in g:
            step_result["saliency"] = g["saliency"].cpu().numpy()
        save_data[i] = step_result

    for tind in range(0, i0.shape[1], 50):
        for t, g in list(zip(ts, gens))[::2]:
            lbl = "DDPM" if ddpm else "DDIM"
            plt.plot(np.squeeze(g["sample"].cpu().numpy())[:, tind], label=f"{lbl}, T={t}")
        plt.plot(np.squeeze(i0)[:, tind], label="Original", linewidth=3, color="black", linestyle="--")
        plt.legend()

        plt.title("Reconstructed Waveform, I: %i" % tind)
        plt.savefig(basedir + "reco_wavf_I%i.png" % tind, bbox_inches="tight")
        plt.close()

    plt.plot(np.abs(np.fft.rfft(np.squeeze(i0), axis=0)).sum(axis=1)[1:], label="Original")
    for t, g in list(zip(ts, gens))[::2]:
        plt.plot(np.abs(np.fft.rfft(np.squeeze(g["pred_xstart"].cpu().numpy()), axis=0)).sum(axis=1)[1:], label=f"DDIM, T={t}")

    plt.legend()
    plt.title("Time-Direction Noise Spectrum")
    plt.savefig(basedir + "time_ffts.png", bbox_inches="tight")
    plt.close()

    plt.plot(np.abs(np.fft.rfft(np.squeeze(i0), axis=1)).sum(axis=0)[1:], label="Original")
    for t, g in list(zip(ts, gens))[::2]:
        plt.plot(np.abs(np.fft.rfft(np.squeeze(g["pred_xstart"].cpu().numpy()), axis=1)).sum(axis=0)[1:], label=f"DDIM, T={t}")

    plt.legend()
    plt.title("Wire-Direction Noise Spectrum")
    plt.savefig(basedir + "wire_ffts.png", bbox_inches="tight")
    plt.close()

    loop = list(
        genf(
            model,
            i0.shape,
            time=ts[-1].item(),
            noise=idiffs[-1].unsqueeze(0),
            cond_fn=cond_fn,
            model_kwargs=model_kwargs,
        )
    )

    for i, img in enumerate(loop[49::50]):
        plt.figure()
        plt.imshow(np.squeeze(img["sample"].cpu().numpy()), vmin=-1, vmax=1)
        plt.colorbar()
        plt.title("Reconstruction Series, Step: %i" % (i * 50 + 49))
        plt.savefig(basedir + "reco_img_t%i_to_%i.png" % (int(ts[-1]), i * 50 + 49), bbox_inches="tight")
        plt.close()

    for tind in range(0, i0.shape[1], 50):
        arr = [np.squeeze(img["sample"].cpu().numpy())[:, tind] for img in loop]
        arr = np.array(arr)
        plt.imshow(arr.T, vmin=-1, vmax=1)
        plt.colorbar()
        plt.title("Reconstruction Series, I: %i" % (tind))
        plt.savefig(basedir + "reco_loop_I%i.png" % tind, bbox_inches="tight")
        plt.close()

    return save_data

In [ ]:
import pickle
basedir = "anomaly-detection/outputs/"
os.makedirs(basedir, exist_ok=True)

In [ ]:
model

In [ ]:
# without classifier
save_data = {}

arr_reco, arr_truth, weights, pixel_weights, charge = load_images(path, show_plot=False)
iidxs = [11, 14, 29, 76, 77, 103, 128]
for img_idx in tqdm(iidxs[:1]):
    # TODO: apply weights if used in training
    this_img = arr_reco[img_idx]
    # arr_reco[img_idx] is [C,H,W]; add batch dim -> [1,C,H,W]
    this_img = th.tensor(this_img, device=dist_util.dev()) #.unsqueeze(0).unsqueeze(0)
    with th.no_grad():
        save_data[img_idx] = validation_plots(diffusion, model, basedir, this_img, ddpm=True)

with open(basedir + "translation_output_woclass.pkl", "wb") as f:
    pickle.dump(save_data, f)

In [ ]:
# without classifier
save_data = {}

arr_reco, arr_truth, weights, pixel_weights, charge = load_images(path, show_plot=False)
iidxs = [11, 14, 29, 76, 77, 103, 128]
for img_idx in tqdm(iidxs[:1]):
    # TODO: apply weights if used in training
    this_img = arr_reco[img_idx]
    # arr_reco[img_idx] is [C,H,W]; add batch dim -> [1,C,H,W]
    this_img = th.tensor(this_img, device=dist_util.dev()).unsqueeze(0)
    save_data[img_idx] = validation_plots_wclass(diffusion, model, basedir, this_img, ddpm=True)

with open(basedir + "translation_output_woclass.pkl", "wb") as f:
    pickle.dump(save_data, f)

In [ ]:
# without classifier
save_data = {}

arr_reco, arr_truth, weights, pixel_weights, charge = load_images(path, show_plot=False)
iidxs = [11, 14, 29, 76, 77, 103, 128]
for img_idx in tqdm(iidxs[:1]):
    # TODO: apply weights if used in training
    this_img = arr_reco[img_idx]
    # arr_reco[img_idx] is [C,H,W]; add batch dim -> [1,C,H,W]
    this_img = th.tensor(this_img, device=dist_util.dev()).unsqueeze(0)
    save_data[img_idx] = validation_plots_wclass(diffusion, model, basedir, this_img, ddpm=True)

with open(basedir + "translation_output_woclass.pkl", "wb") as f:
    pickle.dump(save_data, f)

In [ ]:
# with classifier
save_data = {}

arr_reco, arr_truth, weights, pixel_weights, charge = load_images(path, show_plot=False)
# interesting_idxs = [38, 39, 24, 25, 29]
iidxs = [11, 14, 29, 76, 77, 103, 128]
for img_idx in tqdm(iidxs[:1]):
    # TODO: apply weights if used in training
    this_img = arr_reco[img_idx]
    # arr_reco[img_idx] is [C,H,W]; add batch dim -> [1,C,H,W]
    this_img = th.tensor(this_img, device=dist_util.dev()).unsqueeze(0)
    save_data[img_idx] = validation_plots_wclass(
        diffusion,
        model,
        basedir,
        this_img,
        ddpm=True,
        classifier=model_classifier,
        classifier_scale=100.0,
        target_class=0,
    )

with open(basedir + "translation_output_wclass.pkl", "wb") as f:
    pickle.dump(save_data, f)

## Inspect outputs

In [ ]:
basedir = "/exp/sbnd/app/users/munjung/anomaly-detection/translation/"
iidx = 11
t = 2

# without classifier
with open(basedir + "translation_output_woclass.pkl", "rb") as f:
    save_data_woclass = pickle.load(f)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
img_orig = save_data_woclass[iidx]["original"]
ax[0].imshow(img_orig, vmin=-1, vmax=1, cmap="bwr")
ax[0].title.set_text("Original")

img_diffused = save_data_woclass[iidx][t]["diffused"]
ax[1].imshow(img_diffused[0], vmin=-0.3, vmax=0.3, cmap="bwr")
ax[1].title.set_text("Diffused")

img_reconstructed = save_data_woclass[iidx][t]["reconstructed"]
ax[2].imshow(img_reconstructed[0][0], vmin=-1, vmax=1, cmap="bwr")
ax[2].title.set_text("Reconstructed")

img_diff = img_reconstructed[0][0] - img_orig
ax[3].imshow(img_diff, vmin=-0.3, vmax=0.3, cmap="bwr")
ax[3].title.set_text("Reconstructed - Original")

fig.colorbar(ax[0].images[0], ax=ax, orientation="vertical", fraction=0.04)
plt.show()

# with classifier
with open(basedir + "translation_output_wclass.pkl", "rb") as f:
    save_data_wclass = pickle.load(f)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
img_orig = save_data_wclass[iidx]["original"]
ax[0].imshow(img_orig, vmin=-1, vmax=1, cmap="bwr")
ax[0].title.set_text("Original")

img_diffused = save_data_wclass[iidx][t]["diffused"]
ax[1].imshow(img_diffused[0], vmin=-0.3, vmax=0.3, cmap="bwr")
ax[1].title.set_text("Diffused")

img_reconstructed = save_data_wclass[iidx][t]["reconstructed"]
ax[2].imshow(img_reconstructed[0][0], vmin=-1, vmax=1, cmap="bwr")
ax[2].title.set_text("Reconstructed")

img_diff = img_reconstructed[0][0] - img_orig
ax[3].imshow(img_diff, vmin=-0.3, vmax=0.3, cmap="bwr")
ax[3].title.set_text("Reconstructed - Original")

fig.colorbar(ax[0].images[0], ax=ax, orientation="vertical", fraction=0.04)
plt.show()

### loss gradient as saliency maps

In [ ]:
# make anomaly (saliency) map from the gradient (as opposed to the subtraction map like above)

def classifier_loss_gradient_map(image_np, classifier, target_class=0, timestep=0):
    x = th.tensor(image_np, dtype=th.float32, device=dist_util.dev()).unsqueeze(0).unsqueeze(0)
    x = x.clone().detach().requires_grad_(True)

    t_in = th.full((x.shape[0],), timestep, device=x.device, dtype=th.long)
    y = th.full((x.shape[0],), target_class, device=x.device, dtype=th.long)

    classifier.zero_grad(set_to_none=True)
    logits = classifier(x, t_in)
    loss = F.cross_entropy(logits, y)
    loss.backward()

    grad = x.grad.detach().cpu().numpy()[0, 0]
    saliency = np.abs(grad)
    saliency = saliency / (saliency.max() + 1e-12)
    return saliency, float(loss.detach().cpu().item())

def diffusion_loss_gradient_map(
    image_np,
    model,
    diffusion,
    n_mc=32,
    t_low=1,
    t_high=600,
):
    # this should point to where the diffusion model is hardest to explain
    # TODO: I think? the diffusion model case is slightly more confusing than the classifier case
    # estimate as |dL_diff/dx|
    # use noise-prediction MSE, averaged over random timesteps and random noise draws

    x0 = th.tensor(image_np, dtype=th.float32, device=dist_util.dev()).unsqueeze(0).unsqueeze(0)
    x0 = x0.clone().detach().requires_grad_(True)

    model.zero_grad(set_to_none=True)

    total_loss = 0.0
    t_low = max(0, int(t_low))
    t_high = min(int(t_high), diffusion.num_timesteps)

    # TODO
    for _ in range(n_mc):
        tt = th.randint(t_low, t_high, (x0.shape[0],), device=x0.device, dtype=th.long)
        eps = th.randn_like(x0)

        xt = diffusion.q_sample(x_start=x0, t=tt, noise=eps)
        pred = model(xt, tt)

        # Support checkpoints that predict both mean/variance by keeping noise channels.
        if pred.shape[1] == 2 * x0.shape[1]:
            pred = pred[:, : x0.shape[1], ...]

        total_loss = total_loss + F.mse_loss(pred, eps)

    loss = total_loss / float(n_mc)
    loss.backward()

    grad = x0.grad.detach().cpu().numpy()[0, 0]
    saliency = np.abs(grad)
    saliency = saliency / (saliency.max() + 1e-12)
    return saliency, float(loss.detach().cpu().item())


In [ ]:
basedir = "/exp/sbnd/app/users/munjung/anomaly-detection/translation/"
iidx = 11
t = 2  # reconstruction index to analyze

with open(basedir + "translation_output_wclass.pkl", "rb") as f:
    save_data_wclass = pickle.load(f)

img_orig = save_data_wclass[iidx]["original"]
img_reco = save_data_wclass[iidx][t]["reconstructed"][0][0]

saliency_map, cls_loss = classifier_loss_gradient_map(
    img_reco,
    model_classifier,
    target_class=0,
    timestep=0,
)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].imshow(img_orig, vmin=-1, vmax=1, cmap="bwr")
ax[0].set_title("Original")

ax[1].imshow(img_reco, vmin=-1, vmax=1, cmap="bwr")
ax[1].set_title("Reconstructed (guided)")

ax[2].imshow(saliency_map, cmap="inferno")
ax[2].set_title(f"Loss-Gradient Map |dL/dx|\nCE loss={cls_loss:.4f}")

ax[3].imshow(img_reco, vmin=-1, vmax=1, cmap="gray")
ax[3].imshow(saliency_map, cmap="inferno", alpha=0.5)
ax[3].set_title("Overlay")

for a in ax:
    a.set_xticks([])
    a.set_yticks([])

plt.tight_layout()
plt.show()


saliency_diff, diff_loss = diffusion_loss_gradient_map(
    img_reco,
    model,
    diffusion,
    n_mc=32,
    t_low=1,
    t_high=600,
)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].imshow(img_orig, vmin=-1, vmax=1, cmap="bwr")
ax[0].set_title("Original")

ax[1].imshow(img_reco, vmin=-1, vmax=1, cmap="bwr")
ax[1].set_title("Reconstructed (guided)")

ax[2].imshow(saliency_diff, cmap="inferno")
ax[2].set_title(f"Diffusion Loss-Gradient |dL_diff/dx|\nMC MSE={diff_loss:.4f}")

ax[3].imshow(img_reco, vmin=-1, vmax=1, cmap="gray")
ax[3].imshow(saliency_diff, cmap="inferno", alpha=0.5)
ax[3].set_title("Overlay")

for a in ax:
    a.set_xticks([])
    a.set_yticks([])

plt.tight_layout()
plt.show()